In [1]:
import pandas as pd            # put all python environment here
import pyarrow as pa
import polars as pl
from pathlib import Path
import numpy as np

In [2]:
for ext in ['pandas.period', 'pandas.interval']:
    try:
        pa.unregister_extension_type(ext)
    except Exception:
        # We use a broad Exception here so it completely ignores 
        # any PyArrow complaints if the type isn't found
        pass

In [3]:
df_sample_s = pd.read_parquet("data/submissions_2024-04.parquet", engine="fastparquet")
print(df_sample_s.dtypes) 
display(df_sample_s.head(5).to_dict(orient="list"))
display(df_sample_s)

author         object
subreddit      object
count_posts     int64
dtype: object


{'author': ['ckimball6588',
  'NBA_MOD',
  'pop-hon_ula',
  'Emperors_Enterprise',
  'Remarkable_Prompt691'],
 'subreddit': ['funny', 'nba', 'femalefashionadvice', 'canada', 'facepalm'],
 'count_posts': [1, 1, 1, 1, 1]}

,author,subreddit,count_posts
0,ckimball6588,funny,1
1,NBA_MOD,nba,1
2,pop-hon_ula,femalefashionadvice,1
3,Emperors_Enterprise,canada,1
4,Remarkable_Prompt691,facepalm,1
...,...,...,...
2285592,lilith799,oddlyspecific,1
2285593,jenrencri,parenting,1
2285594,blurppyyy,mildlyinfuriating,1
2285595,Technology_Hero,memes,1


In [4]:
df_sample_c = pd.read_parquet("data/comments_2024-04.parquet", engine="fastparquet")
print(df_sample_c.dtypes) 
display(df_sample_c.head(5).to_dict(orient="list"))
display(df_sample_c)

author            object
subreddit         object
count_comments     int64
dtype: object


{'author': ['SapphicLullaby',
  'nimmin13',
  '248kb',
  'Milqy',
  'InfiniteLuxGiven'],
 'subreddit': ['crochet',
  'music',
  'mildlyinfuriating',
  'ufos',
  'unitedkingdom'],
 'count_comments': [1, 1, 1, 1, 1]}

,author,subreddit,count_comments
0,SapphicLullaby,crochet,1
1,nimmin13,music,1
2,248kb,mildlyinfuriating,1
3,Milqy,ufos,1
4,InfiniteLuxGiven,unitedkingdom,1
...,...,...,...
43923931,bordomsdeadly,baseball,1
43923932,SpiritualRide528,jujutsukaisen,1
43923933,StripedBadger,harrypotter,1
43923934,WilsonValdro,marvelmemes,1


In [5]:
# AGGREGATION YOU HEAR ME? AGGREGATING !

# analytics/user_subreddit_interactions.parquet 
# with (author, subreddit, interaction_count) 
# aggregated over Jan–Jun yas koriq

# i also generated PER-MONTHs JIC YK... IDK
# but for the next codee, we'r j gna use the agg'd 6-mos 

In [6]:
# established file paths

PARQUET_DIR = Path("data")
OUT_DIR = Path("analytics")
OUT_DIR.mkdir(exist_ok=True)

MONTHS = ["01", "02", "03", "04", "05", "06"]  # changed this part from march -> 03
OVERWRITE = False

In [7]:
# functions for reading parquet and cleaning them 

def safe_read_parquet(path: Path) -> pl.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pl.read_parquet(path)

def clean_interactions(df: pl.DataFrame, count_col: str) -> pl.DataFrame:
    return (
        df
        .filter(
            pl.col("author").is_not_null()
            & pl.col("subreddit").is_not_null()
            & (pl.col("author") != "[deleted]")
            & (pl.col("author") != "AutoModerator")
        )
        .with_columns(
            pl.col("subreddit").str.to_lowercase(),
            pl.col(count_col).cast(pl.Int64),
        )
    )

def safe_write_parquet(df: pl.DataFrame, path: Path):
    if path.exists() and not OVERWRITE:
        print(f"[skip] {path} already exists")
        return
    df.write_parquet(path)
    print(f"[write] {path}")

def load_month_interactions(month: str) -> pl.DataFrame:
    sub_path = PARQUET_DIR / f"submissions_2024-{month}.parquet"     # changed this part so that wont have to dl seprate parquet files
    com_path = PARQUET_DIR / f"comments_2024-{month}.parquet"

    print(f"[load] {month} submissions: {sub_path}")
    df_posts = safe_read_parquet(sub_path)
    print(f"[load] {month} comments  : {com_path}")
    df_comments = safe_read_parquet(com_path)

    # clean posts - display indiv
    df_posts = (
        clean_interactions(df_posts, "count_posts")
        .group_by(["author", "subreddit"])
        .agg(pl.col("count_posts").sum())
    )

    # clean comments - display indiv
    df_comments = (
        clean_interactions(df_comments, "count_comments")
        .group_by(["author", "subreddit"])
        .agg(pl.col("count_comments").sum())
    )

    # merge posts and comments
    df_month = df_posts.join(
        df_comments,
        on=["author", "subreddit"],
        how="full",
        coalesce=True
    ).fill_null(0)

    # total interactions
    df_month = df_month.with_columns(
        (pl.col("count_posts") + pl.col("count_comments")).alias("interaction_count")
    )

    return df_month

In [8]:
# run the functions

# streaming-style global aggregation to control memory
df_all = None

for m in MONTHS:
    print(f"[month] {m}")
    df_m = load_month_interactions(m)

    # optional per-month checkpoint
    month_out = OUT_DIR / f"user_subreddit_2024_{m}.parquet"
    safe_write_parquet(df_m, month_out)

    if df_all is None:
        df_all = df_m
    else:
        df_all = (
            pl.concat([df_all, df_m])
            .group_by(["author", "subreddit"])
            .agg([
                pl.col("count_posts").sum().cast(pl.Int64),
                pl.col("count_comments").sum().cast(pl.Int64),
                pl.col("interaction_count").sum().cast(pl.Int64),
            ])
        )

final_out = OUT_DIR / "user_subreddit_interactions.parquet"
safe_write_parquet(df_all, final_out)

[month] 01
[load] 01 submissions: data/submissions_2024-01.parquet
[load] 01 comments  : data/comments_2024-01.parquet
[skip] analytics/user_subreddit_2024_01.parquet already exists
[month] 02
[load] 02 submissions: data/submissions_2024-02.parquet
[load] 02 comments  : data/comments_2024-02.parquet
[skip] analytics/user_subreddit_2024_02.parquet already exists
[month] 03
[load] 03 submissions: data/submissions_2024-03.parquet
[load] 03 comments  : data/comments_2024-03.parquet
[skip] analytics/user_subreddit_2024_03.parquet already exists
[month] 04
[load] 04 submissions: data/submissions_2024-04.parquet
[load] 04 comments  : data/comments_2024-04.parquet
[skip] analytics/user_subreddit_2024_04.parquet already exists
[month] 05
[load] 05 submissions: data/submissions_2024-05.parquet
[load] 05 comments  : data/comments_2024-05.parquet
[skip] analytics/user_subreddit_2024_05.parquet already exists
[month] 06
[load] 06 submissions: data/submissions_2024-06.parquet
[load] 06 comments  : d

In [9]:
# inspect aggregated dataframe

df_all
# print(df_all.shape)

author,subreddit,count_posts,count_comments,interaction_count
str,str,i64,i64,i64
"""Thisismyuserok""","""funnysigns""",0,1,1
"""Chedskiee""","""running""",1,0,1
"""narmowen""","""pettyrevenge""",0,1,1
"""Commie_creator""","""books""",0,4,4
"""mg1120""","""wallstreetbets""",0,10,10
…,…,…,…,…
"""mpking828""","""buyitforlife""",0,1,1
"""TFC268""","""askreddit""",0,6,6
"""Super_Wario_128""","""forbiddensnacks""",1,0,1


In [10]:
# SELECTING IDENTIFYING THEEE TOP 500 SUBREDDITS & ACTIVE USERS

# output for ths are:

## table for top 500 subreddits
### sub_activity: (subreddit, total_interactions) sorted descending

## table for filtered interaction
### df_active: all (author, subreddit, interaction_count) rows
#### where subreddit is an elements of the top 500 ;
#### where author is an active user (in df_active) :
##### user_total_interactions >= 20 (still an elements of the top 500);
##### user_n_subs >= 2 (multi‑community)

# GETS? lord    ok gets 


In [11]:
ANALYTICS_DIR = Path("analytics")
INTERACTIONS_PATH = ANALYTICS_DIR / "user_subreddit_interactions.parquet"
ACTIVE_PATH = ANALYTICS_DIR / "user_subreddit_active_interactions.parquet"

df_full = pl.read_parquet(INTERACTIONS_PATH)

# lord ito quick sanity check 
print("Full interactions table:", df_full.shape)

Full interactions table: (46724134, 5)


In [12]:
# Top 500 subreddits by total interactions
sub_activity = (
    df_full
    .group_by("subreddit")
    .agg(pl.col("interaction_count").sum().alias("total_interactions"))
    .sort("total_interactions", descending=True)
)

# print("sub_activity shape:", sub_activity.shape)
# print("Top 10 subreddits:")
# print(sub_activity.head(10))

# # only taking the 500 most active subreddits
# top500 = sub_activity.head(500)

# # siguraduhin na may saktong 500 unique subreddit names
# top500_subs = top500["subreddit"].unique().to_list()
# print("Unique subreddits in top500 list:", len(top500_subs))
# assert len(top500_subs) == 500, "Top500 must contain 500 unique subreddits"

# # now filter df to only those 500 subreddits
# df_topsubs = df.join(
#     pl.DataFrame({"subreddit": top500_subs}),
#     on="subreddit",
#     how="inner",
# )

# print("Unique subreddits in df:", df["subreddit"].n_unique())
# print("Unique subreddits in df_topsubs:", df_topsubs["subreddit"].n_unique())
# print("After restricting to top 500 subs:", df_topsubs.shape)

print("Total subreddits:", sub_activity.height)  # dapat 500 yan lord
print("Top 10 subreddits:\n", sub_activity.head(10))

Total subreddits: 500
Top 10 subreddits:
 shape: (10, 2)
┌───────────────────┬────────────────────┐
│ subreddit         ┆ total_interactions │
│ ---               ┆ ---                │
│ str               ┆ i64                │
╞═══════════════════╪════════════════════╡
│ askreddit         ┆ 21332107           │
│ aitah             ┆ 7301950            │
│ facepalm          ┆ 5686680            │
│ nostupidquestions ┆ 5379142            │
│ teenagers         ┆ 5204826            │
│ nba               ┆ 5058636            │
│ helldivers        ┆ 4796214            │
│ wallstreetbets    ┆ 4452225            │
│ politics          ┆ 4002790            │
│ nfl               ┆ 4001359            │
└───────────────────┴────────────────────┘


In [13]:
user_stats = (
    df_full
    .group_by("author")
    .agg([
        pl.col("interaction_count").sum().alias("user_total_interactions"),
        pl.col("subreddit").n_unique().alias("user_n_subs"),
    ])
)

In [14]:
# ang median total interactions is 3 lang; median number of subreddits is 1
# mean total interaction is apprx 20; it still include a very large fraction

# new filters: >= 30 interactions in >= 3 subreddits

# active users count = 1,366,020 (about 10% of all 13.36M users)


# reasoning for 30: above normal sha (2x of mean, and far above the 75th %ile),

# Global median interactions = 3; 75th percentile = 9
# with >= 30 interactions and >= 3 subreddits:
 ## active users count = 1,366,020 (about 10% of all 13.36M users)
 ## median interactions among actives = 71 (vs 3 globally)
 ## median subreddits among actives = 12 (vs 1 globally)

In [15]:
ACTIVE_MIN_INTERACTIONS = 30
ACTIVE_MIN_SUBS = 3

active_users = user_stats.filter(
    (pl.col("user_total_interactions") >= ACTIVE_MIN_INTERACTIONS)
    & (pl.col("user_n_subs") >= ACTIVE_MIN_SUBS)
)

active_user_ids = set(active_users["author"].to_list())
print("Active users count:", len(active_user_ids))
print(active_users.select("user_total_interactions").describe())
print(active_users.select("user_n_subs").describe())

df_active = df_full.filter(pl.col("author").is_in(active_user_ids))
print("df_active shape after enforcing filters:", df_active.shape)

Active users count: 1366020
shape: (9, 2)
┌────────────┬─────────────────────────┐
│ statistic  ┆ user_total_interactions │
│ ---        ┆ ---                     │
│ str        ┆ f64                     │
╞════════════╪═════════════════════════╡
│ count      ┆ 1.36602e6               │
│ null_count ┆ 0.0                     │
│ mean       ┆ 143.681262              │
│ std        ┆ 261.762073              │
│ min        ┆ 30.0                    │
│ 25%        ┆ 43.0                    │
│ 50%        ┆ 71.0                    │
│ 75%        ┆ 143.0                   │
│ max        ┆ 39271.0                 │
└────────────┴─────────────────────────┘
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ user_n_subs │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 1.36602e6   │
│ null_count ┆ 0.0         │
│ mean       ┆ 15.417882   │
│ std        ┆ 11.923861   │
│ min        ┆ 3.0         │
│ 25%        ┆ 7.0         │
│ 50%     

In [16]:
ACTIVE_PATH = ANALYTICS_DIR / "user_subreddit_active_interactions.parquet"
df_active.write_parquet(ACTIVE_PATH)
print(f"Saved enforced-active interactions to: {ACTIVE_PATH}")

Saved enforced-active interactions to: analytics/user_subreddit_active_interactions.parquet


In [17]:
# USER FEATURE CONSTRUCTION FOR CLUSTERING


In [18]:
ANALYTICS_DIR = Path("analytics")
ACTIVE_PATH = ANALYTICS_DIR / "user_subreddit_active_interactions.parquet"
USER_FEATS_PATH = ANALYTICS_DIR / "user_features_for_clustering.parquet"

In [19]:
# load df_active
df_active = pl.read_parquet(ACTIVE_PATH)
print("[3.1] Active interactions table shape:", df_active.shape)

[3.1] Active interactions table shape: (21061135, 5)


In [20]:
# per-user totals and spread
user_totals = (
    df_active
    .group_by("author")
    .agg([
        pl.col("interaction_count").sum().alias("user_total_interactions"),
        pl.col("subreddit").n_unique().alias("user_n_subs"),
    ])
)

print("[3.2] user_totals shape:", user_totals.shape)
print(user_totals.select("user_total_interactions").describe())
print(user_totals.select("user_n_subs").describe())


[3.2] user_totals shape: (1366020, 3)
shape: (9, 2)
┌────────────┬─────────────────────────┐
│ statistic  ┆ user_total_interactions │
│ ---        ┆ ---                     │
│ str        ┆ f64                     │
╞════════════╪═════════════════════════╡
│ count      ┆ 1.36602e6               │
│ null_count ┆ 0.0                     │
│ mean       ┆ 143.681262              │
│ std        ┆ 261.762073              │
│ min        ┆ 30.0                    │
│ 25%        ┆ 43.0                    │
│ 50%        ┆ 71.0                    │
│ 75%        ┆ 143.0                   │
│ max        ┆ 39271.0                 │
└────────────┴─────────────────────────┘
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ user_n_subs │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 1.36602e6   │
│ null_count ┆ 0.0         │
│ mean       ┆ 15.417882   │
│ std        ┆ 11.923861   │
│ min        ┆ 3.0         │
│ 25%        ┆ 7.0         │


In [21]:
# per-user per-subreddit shares
user_sub_shares = (
    df_active
    .group_by(["author", "subreddit"])
    .agg(pl.col("interaction_count").sum().alias("cnt"))
)

user_sub_shares = user_sub_shares.join(
    user_totals.select(["author", "user_total_interactions"]),
    on="author",
    how="left",
)

user_sub_shares = user_sub_shares.with_columns(
    (pl.col("cnt") / pl.col("user_total_interactions")).alias("p")
)

print("[3.3] user_sub_shares shape:", user_sub_shares.shape)
print(user_sub_shares.head())

[3.3] user_sub_shares shape: (21061135, 5)
shape: (5, 5)
┌────────────────┬───────────────────┬─────┬─────────────────────────┬──────────┐
│ author         ┆ subreddit         ┆ cnt ┆ user_total_interactions ┆ p        │
│ ---            ┆ ---               ┆ --- ┆ ---                     ┆ ---      │
│ str            ┆ str               ┆ i64 ┆ i64                     ┆ f64      │
╞════════════════╪═══════════════════╪═════╪═════════════════════════╪══════════╡
│ wilderkin1     ┆ teenagers         ┆ 2   ┆ 242                     ┆ 0.008264 │
│ Wick0158       ┆ homeowners        ┆ 1   ┆ 37                      ┆ 0.027027 │
│ chettyoubetcha ┆ mildlyinfuriating ┆ 1   ┆ 66                      ┆ 0.015152 │
│ SonJake21      ┆ mademesmile       ┆ 1   ┆ 182                     ┆ 0.005495 │
│ KooshIsKing    ┆ baldursgate3      ┆ 135 ┆ 328                     ┆ 0.411585 │
└────────────────┴───────────────────┴─────┴─────────────────────────┴──────────┘


In [22]:
# sanity check - same as the one in basic goals ipynb
testing = user_sub_shares.filter(
    (pl.col("author") == "MajesticRocket") & 
    (pl.col("subreddit") == "canada")
)

testing

author,subreddit,cnt,user_total_interactions,p
str,str,i64,i64,f64
"""MajesticRocket""","""canada""",1,63,0.015873


In [23]:
# aggregate to max_share and entropy per user

def entropy_from_probs(probs):
    arr = np.asarray(probs, dtype=float)
    arr = arr[arr > 0]
    if arr.size == 0:
        return 0.0
    return float(-(arr * np.log(arr)).sum())

# group and collect shares into list per user
user_shares_agg = (
    user_sub_shares
    .group_by("author")
    .agg([
        pl.col("p").max().alias("max_share"),
        pl.col("p").alias("p_list"),   # list[f64] per user
    ])
)

print("[3.4] user_shares_agg shape:", user_shares_agg.shape)
print(user_shares_agg.head())

# move to Python, compute entropy, and rebuild a Polars DF
pdf = user_shares_agg.to_pandas()
pdf["entropy"] = pdf["p_list"].apply(entropy_from_probs)
pdf = pdf.drop(columns=["p_list"])

user_shares_agg = pl.from_pandas(pdf)

print("[3.4b] user_shares_agg with entropy:")
print(user_shares_agg.head())

[3.4] user_shares_agg shape: (1366020, 3)
shape: (5, 3)
┌───────────────────┬───────────┬─────────────────────────────────┐
│ author            ┆ max_share ┆ p_list                          │
│ ---               ┆ ---       ┆ ---                             │
│ str               ┆ f64       ┆ list[f64]                       │
╞═══════════════════╪═══════════╪═════════════════════════════════╡
│ incywince         ┆ 0.489231  ┆ [0.006154, 0.009231, … 0.00615… │
│ Dazzling_Pea5290  ┆ 0.666667  ┆ [0.041667, 0.229167, … 0.0625]  │
│ ElopedCantelope   ┆ 0.222222  ┆ [0.044444, 0.088889, … 0.08888… │
│ SteveYzerman_19   ┆ 0.691489  ┆ [0.010638, 0.021277, … 0.11702… │
│ IMadeThis4HOIMods ┆ 0.43871   ┆ [0.070968, 0.43871, … 0.032258… │
└───────────────────┴───────────┴─────────────────────────────────┘
[3.4b] user_shares_agg with entropy:
shape: (5, 3)
┌───────────────────┬───────────┬──────────┐
│ author            ┆ max_share ┆ entropy  │
│ ---               ┆ ---       ┆ ---      │
│ str     

In [24]:
# sanity check - same as the one in basic goals ipynb
testing_2 = user_shares_agg.filter(
    (pl.col("author") == "NineteenSixtySix") & 
    (pl.col("max_share").is_between(0.097221, 0.097223))
)

testing_2

author,max_share,entropy
str,f64,f64
"""NineteenSixtySix""",0.097222,3.400263


In [25]:
user_features = (
    user_totals
    .join(user_shares_agg, on="author", how="left")
    .with_columns(
        pl.col("user_total_interactions")
        .cast(pl.Float64)
        .log()
        .alias("log_total_interactions")
    )
)

print("[3.5] user_features shape:", user_features.shape)
print(user_features.head())

[3.5] user_features shape: (1366020, 6)
shape: (5, 6)
┌──────────────────┬─────────────────────┬─────────────┬───────────┬──────────┬────────────────────┐
│ author           ┆ user_total_interact ┆ user_n_subs ┆ max_share ┆ entropy  ┆ log_total_interact │
│ ---              ┆ ions                ┆ ---         ┆ ---       ┆ ---      ┆ ions               │
│ str              ┆ ---                 ┆ u32         ┆ f64       ┆ f64      ┆ ---                │
│                  ┆ i64                 ┆             ┆           ┆          ┆ f64                │
╞══════════════════╪═════════════════════╪═════════════╪═══════════╪══════════╪════════════════════╡
│ MK_Scorpion      ┆ 239                 ┆ 4           ┆ 0.92887   ┆ 0.297688 ┆ 5.476464           │
│ NYState_of_Mind  ┆ 243                 ┆ 6           ┆ 0.699588  ┆ 0.933983 ┆ 5.493061           │
│ sokos            ┆ 96                  ┆ 4           ┆ 0.895833  ┆ 0.432408 ┆ 4.564348           │
│ Ok-Painting-1782 ┆ 62              

In [26]:
USER_FEATS_PATH = ANALYTICS_DIR / "user_features_for_clustering.parquet"
user_features.write_parquet(USER_FEATS_PATH)
print(f"[3.6] Saved user features to: {USER_FEATS_PATH}")

[3.6] Saved user features to: analytics/user_features_for_clustering.parquet


In [27]:
# CCLUSTERING ON USER BEHAVIORAL FEATURES

In [28]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [29]:
ANALYTICS_DIR = Path("analytics")
USER_FEATS_PATH = ANALYTICS_DIR / "user_features_for_clustering.parquet"
CLUSTERS_PATH = ANALYTICS_DIR / "user_clusters.parquet"

In [30]:
#LOAD features

user_features = pl.read_parquet(USER_FEATS_PATH)
print("[4.1] user_features shape:", user_features.shape)

[4.1] user_features shape: (1366020, 6)


In [31]:
# select feature matrix

feature_cols = ["log_total_interactions", "user_n_subs", "entropy"]
X = user_features.select(feature_cols).to_numpy()

In [32]:
# Standardize

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("[4.3] X_scaled shape:", X_scaled.shape)


[4.3] X_scaled shape: (1366020, 3)


In [33]:
# clustering K = {3, 4, 5} avoids underfitting (1-2 clus), and overfragmentation (6+ clus)

# many case studies also focus on small K (often 3-5)

# https://akanshasaxena.com/challenge/ml/30-days-30-ml-projects-day-24/  ----- this use k=5
# https://www.kaggle.com/code/debjeetdas/customer-segmentation-using-kmeans-clustering  ----- k=5

In [34]:
# choose K in {3,4,5} by silhouette

Ks = [3, 4, 5]
silhouettes = []

for k in Ks:
    labels = KMeans(n_clusters=k, random_state=42, n_init="auto").fit_predict(X_scaled)
    silhouettes.append(silhouette_score(X_scaled, labels))


best_k = Ks[int(np.argmax(silhouettes))]
print(dict(zip(Ks, silhouettes)))
print("best_k:", best_k)

{3: 0.34806379755834416, 4: 0.37346815922249965, 5: 0.35058884425363}
best_k: 4


In [35]:
# refit on best_k and attach labels
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init="auto")
user_clusters = user_features.with_columns(
    pl.Series("cluster", kmeans.fit_predict(X_scaled))
)

In [36]:
print("[4.6] user_clusters shape:", user_clusters.shape)
print(user_clusters.head())

[4.6] user_clusters shape: (1366020, 7)
shape: (5, 7)
┌─────────────────┬────────────────┬─────────────┬───────────┬──────────┬────────────────┬─────────┐
│ author          ┆ user_total_int ┆ user_n_subs ┆ max_share ┆ entropy  ┆ log_total_inte ┆ cluster │
│ ---             ┆ eractions      ┆ ---         ┆ ---       ┆ ---      ┆ ractions       ┆ ---     │
│ str             ┆ ---            ┆ u32         ┆ f64       ┆ f64      ┆ ---            ┆ i32     │
│                 ┆ i64            ┆             ┆           ┆          ┆ f64            ┆         │
╞═════════════════╪════════════════╪═════════════╪═══════════╪══════════╪════════════════╪═════════╡
│ MK_Scorpion     ┆ 239            ┆ 4           ┆ 0.92887   ┆ 0.297688 ┆ 5.476464       ┆ 1       │
│ NYState_of_Mind ┆ 243            ┆ 6           ┆ 0.699588  ┆ 0.933983 ┆ 5.493061       ┆ 1       │
│ sokos           ┆ 96             ┆ 4           ┆ 0.895833  ┆ 0.432408 ┆ 4.564348       ┆ 3       │
│ Ok-Painting-178 ┆ 62             ┆ 

In [37]:
# cluster profiling
cluster_profile = (
    user_clusters
    .groupby("cluster")
    .agg([
        pl.col("author").count().alias("n_users"),
        pl.col("user_total_interactions").mean().alias("mean_total"),
        pl.col("user_n_subs").mean().alias("mean_n_subs"),
        pl.col("max_share").mean().alias("mean_max_share"),
        pl.col("entropy").mean().alias("mean_entropy"),
    ])
    .sort("cluster")
)
print(cluster_profile)

AttributeError: 'DataFrame' object has no attribute 'groupby'

In [ ]:
# save

CLUSTERS_PATH = ANALYTICS_DIR / "user_clusters.parquet"
user_clusters.write_parquet(CLUSTERS_PATH)
print(f"[4.7] Saved user clusters to: {CLUSTERS_PATH}")